# HW5 — From Predictions to a Portfolio

**FIN 7057 · Due after Week 9 · 100 pts**

Turn cross-sectional predictions into a long-short portfolio and judge it on **net-of-cost**
performance. Fill in the `# TODO`s.


## 0. Setup + provided evaluation helper


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
SEED=3; np.random.seed(SEED); ANN=np.sqrt(252)

def evaluate(weights, fwd_ret, cost_bps=0.0):
    """weights, fwd_ret: (days x assets). weights[t] earns fwd_ret[t].
    Normalizes to gross exposure 1; returns gross/net Sharpe, turnover, max drawdown."""
    w=weights/ (np.abs(weights).sum(1,keepdims=True)+1e-12)
    gross=(w*fwd_ret).sum(1)
    turn=np.r_[0, np.abs(np.diff(w,axis=0)).sum(1)]
    net=gross - cost_bps/1e4*turn
    sh=lambda x: x.mean()/x.std()*ANN if x.std()>0 else np.nan
    curve=np.cumsum(net); dd=(np.maximum.accumulate(curve)-curve).max()
    return {'gross_Sharpe':sh(gross),'net_Sharpe':sh(net),'turnover':turn.mean(),'max_dd':dd,'net_curve':curve}

def mean_ic(pred, fwd):
    return float(np.nanmean([spearmanr(pred[t], fwd[t]).statistic for t in range(len(pred))]))


## Part A — A cross-sectional signal (replace with your model's predictions)


In [ ]:
days,na=900,60
rng=np.random.default_rng(SEED)
latent=rng.standard_normal((days,na))
fwd=0.03*latent+rng.standard_normal((days,na))

def build_predictions(realized_returns):
    # TODO: replace with your point-in-time model. Row t may use only information
    # available before the return in fwd[t] is realized.
    return pd.DataFrame(realized_returns).rolling(60).mean().shift(1).fillna(0.0).to_numpy()

pred=build_predictions(fwd)
print(f'rank IC: {mean_ic(pred,fwd):+.3f}')


## Part B — Scores -> positions (>=2 sizing schemes)


In [ ]:
rank=pd.DataFrame(pred).rank(axis=1).to_numpy()
schemes={
  'quintile L/S':np.where(rank>=0.8*na,1.,0)+np.where(rank<=0.2*na,-1.,0),
  'rank-prop':rank-(na+1)/2,
}
# TODO: two schemes are required. Add another only if it tests a distinct sizing idea.


## Part C & D — cost-aware backtest + comparison table


In [ ]:
rows=[]
for nm,w in schemes.items():
    w=w.astype(float)
    base=evaluate(w,fwd,0)
    rows.append({'scheme':nm,'IC':round(mean_ic(pred,fwd),3),
                 'gross_Sharpe':round(base['gross_Sharpe'],2),
                 'net@10bps':round(evaluate(w,fwd,10)['net_Sharpe'],2),
                 'net@20bps':round(evaluate(w,fwd,20)['net_Sharpe'],2),
                 'turnover':round(base['turnover'],2),
                 'max_dd':round(evaluate(w,fwd,10)['max_dd'],3)})
print(pd.DataFrame(rows).to_string(index=False))
# Equity curve of one scheme:
ec=evaluate(schemes['rank-prop'].astype(float),fwd,10)['net_curve']
plt.figure(figsize=(10,3)); plt.plot(ec); plt.title('Net cumulative long-short return (rank-prop, 10bps)'); plt.tight_layout(); plt.show()
# TODO (Part D, >=200 words): does highest gross == highest net? which would you deploy? 
#   describe a case where higher IC -> worse net portfolio (turnover).


## Part E — Volatility targeting + limitations


In [ ]:
# Vol-target one strategy with a declared target and a past-only volatility estimate.
w=schemes['rank-prop'].astype(float)
w=w/(np.abs(w).sum(1,keepdims=True)+1e-12)
pnl=pd.Series((w*fwd).sum(1))
TARGET_DAILY_VOL=0.01
def build_vol_scale(pnl_series):
    vol=pnl_series.rolling(21,min_periods=21).std().shift(1)
    return (TARGET_DAILY_VOL/(vol+1e-12)).clip(upper=4.0).fillna(0.0)
scale=build_vol_scale(pnl)
vt=scale*pnl
print(f'Std of rolling-21 vol: plain {pnl.rolling(21).std().std():.4f}; '
      f'vol-targeted {vt.rolling(21).std().std():.4f}')
# TODO: explain what changed and discuss capacity, concentration, costs, and regimes.


## Extensions (optional)

- Add no-trade bands and measure the turnover/forecast-decay trade-off.
- Replace linear costs with a size-dependent impact model and report capacity.
- Build a sector- or beta-neutral version and compare concentration and drawdown.


## Self-check (auto-graded)

Run this cell last. It checks alignment, future invariance, required comparisons, and cost
accounting. It deliberately does not require a positive signal or a preferred portfolio ranking.
Written interpretation is graded by hand.


In [ ]:
# ================= SELF-CHECK (auto-graded) =================
def _check(name,fn,hint=""):
    try:
        ok=bool(fn())
    except Exception as e:
        ok,hint=False,hint or f"({type(e).__name__}: {e})"
    print(("PASS  " if ok else "FAIL  ")+name+("" if ok else "   -> "+hint))
    return ok

def _prediction_future_disc():
    base=build_predictions(fwd)
    changed=fwd.copy(); t0=len(fwd)//2
    changed[t0+1:]*=-2
    return float(np.max(np.abs(base[:t0+1]-build_predictions(changed)[:t0+1])))

def _vol_scale_future_disc():
    base=build_vol_scale(pnl)
    changed=pnl.copy(); t0=len(pnl)//2
    changed.iloc[t0+1:]=changed.iloc[t0+1:]*-3+0.1
    return float(np.max(np.abs(base.iloc[:t0+1]-build_vol_scale(changed).iloc[:t0+1])))

_r=[]
_r.append(_check("reproducibility: SEED set",lambda:SEED is not None,"set SEED"))
_r.append(_check("predictions aligned with returns",lambda:np.shape(pred)==np.shape(fwd),"pred and fwd must have the same days x assets shape"))
_r.append(_check("future returns do not alter past predictions",lambda:_prediction_future_disc()<1e-10,"make build_predictions point-in-time"))
_r.append(_check("rank IC is finite",lambda:np.isfinite(mean_ic(pred,fwd)),"compute a finite cross-sectional rank IC; it need not be positive"))
_r.append(_check(">= 2 portfolio schemes compared",lambda:len(schemes)>=2,"compare at least two sizing schemes"))
_r.append(_check("vol target uses a declared finite target",lambda:np.isfinite(TARGET_DAILY_VOL) and TARGET_DAILY_VOL>0,"declare TARGET_DAILY_VOL"))
_r.append(_check("volatility scaling is past-only",lambda:_vol_scale_future_disc()<1e-10,"lag the volatility estimate and do not backfill early rows"))
_r.append(_check("evaluate reports cost-aware metrics",lambda:{'net_Sharpe','turnover','max_dd'}<=set(evaluate(np.ones_like(fwd),fwd,10)),"keep evaluate()"))
_r.append(_check("transaction costs reduce net Sharpe",lambda:evaluate(list(schemes.values())[0].astype(float),fwd,20)['net_Sharpe']<=evaluate(list(schemes.values())[0].astype(float),fwd,0)['net_Sharpe']+1e-9,"check cost alignment"))
print(f"\n{sum(_r)}/{len(_r)} mechanical checks passed.")
assert all(_r),"Fix the FAIL items above before submitting."


## AI-use disclosure & reproducibility checklist
TODO.
